# Baseline: QLoRA Text-to-SVG Generation

**NYU Deep Learning — Spring 2026 Midterm Kaggle Competition**

This notebook implements a complete baseline pipeline in **two phases**:

**Phase A — Training** (requires internet & GPU):

1. Environment Setup — dependencies, seeds, configuration
2. SVG Utilities — competition-compliant validation, post-processing, fallback
3. Data Pipeline — multi-source loading, normalization, quality filtering
4. Model Setup — Qwen 2B + QLoRA via Unsloth
5. Training — SFT with chat-formatted (prompt, SVG) pairs

**Phase B — Inference & Submission** (can run offline on Kaggle):

6. Inference & Submission — generate, validate, export `submission.csv`

> **For Kaggle Code Submission:** split at the phase boundary. Upload the trained adapter as a Kaggle dataset, then create a separate offline notebook that only runs Phase B.

---
## 1. Environment Setup

In [1]:
# Uncomment on a fresh Kaggle / Colab environment
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml cairosvg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 29.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 85.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.9/402.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 70.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━

In [2]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ET.register_namespace('', 'http://www.w3.org/2000/svg')
ET.register_namespace('xlink', 'http://www.w3.org/1999/xlink')

print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
CONFIG = {
    # Model
    'model_name': 'unsloth/Qwen2.5-3B-Instruct',
    'max_seq_length': 2048,

    # LoRA
    'lora_r': 16,
    'lora_alpha': 16,
    'lora_dropout': 0,
    'target_modules': [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],

    # Training
    'learning_rate': 2e-4,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 32,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'logging_steps': 20,
    'eval_steps': 100,
    'save_steps': 200,

    # Data
    'max_train_samples_per_source': 12000,
    'max_svg_train_chars': 2500,
    'eval_size': 0.02,

    # SVG constraints (competition rules)
    'max_svg_length': 16000,
    'max_path_count': 256,
    'canvas_size': 256,

    # Inference
    'inference_batch_size': 4,
    'max_new_tokens': 768,
    'temperature': 0.3,
    'top_p': 0.8,
    'repetition_penalty': 1.05,

    # Paths (adjust for your environment)
    'train_data_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv',
    'output_dir': '/kaggle/working/qwen25_3b_svg_lora',
    'test_prompts_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv',
    'submission_path': '/kaggle/working/submission.csv',
}

CONFIG

{'model_name': 'unsloth/Qwen2.5-3B-Instruct',
 'max_seq_length': 1024,
 'lora_r': 16,
 'lora_alpha': 16,
 'lora_dropout': 0,
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj'],
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 32,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 12000,
 'max_svg_train_chars': 4000,
 'eval_size': 0.02,
 'max_svg_length': 16000,
 'max_path_count': 256,
 'canvas_size': 256,
 'max_new_tokens': 768,
 'temperature': 0.3,
 'top_p': 0.8,
 'repetition_penalty': 1.05,
 'output_dir': '/kaggle/working/qwen25_3b_svg_lora',
 'test_prompts_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv',
 'submission_path': '/kaggle/working/submission.csv'}

---
## 2. SVG Utilities

Competition-compliant validation, post-processing, and fallback.

Key constraints: 256×256 canvas, ≤16 000 chars, ≤256 paths, whitelist-only tags, no scripts/events/animation.

In [4]:
ALLOWED_TAGS = frozenset({
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter',
})


def _strip_ns(tag):
    return tag.split('}', 1)[-1] if '}' in tag else tag


def _count_paths(root):
    return sum(1 for e in root.iter() if _strip_ns(e.tag) == 'path')


def _collect_bad_tags(root):
    return {_strip_ns(e.tag) for e in root.iter() if _strip_ns(e.tag) not in ALLOWED_TAGS}


def _has_event_handlers(root):
    for e in root.iter():
        for attr in e.attrib:
            local = attr.split('}')[-1] if '}' in attr else attr
            if local.lower().startswith('on'):
                return True
    return False


def validate_svg(svg_text, max_length=16000, max_paths=256):
    """
    Competition-compliant SVG validation.
    Returns (is_valid, reason).
    """
    if not svg_text or not isinstance(svg_text, str):
        return False, 'empty'
    if len(svg_text) > max_length:
        return False, f'too_long ({len(svg_text)})'
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError as e:
        return False, f'parse_error: {e}'
    if _strip_ns(root.tag) != 'svg':
        return False, f'root_not_svg ({root.tag})'
    bad = _collect_bad_tags(root)
    if bad:
        return False, f'bad_tags: {bad}'
    if _has_event_handlers(root):
        return False, 'event_handlers'
    n = _count_paths(root)
    if n > max_paths:
        return False, f'too_many_paths ({n})'
    return True, 'ok'


def sanitize_svg(svg_text):
    """
    Post-process SVG: fix root attributes, strip disallowed elements/attributes.
    Returns cleaned SVG string or empty string on failure.
    """
    if not svg_text:
        return ''
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return ''
    if _strip_ns(root.tag) != 'svg':
        return ''
    root.set('xmlns', 'http://www.w3.org/2000/svg')
    root.set('width', '256')
    root.set('height', '256')
    root.set('viewBox', '0 0 256 256')
    _remove_bad_elements(root)
    _remove_event_attrs(root)
    return ET.tostring(root, encoding='unicode')


def _remove_bad_elements(elem):
    to_remove = [c for c in elem if _strip_ns(c.tag) not in ALLOWED_TAGS]
    for c in to_remove:
        elem.remove(c)
    for c in elem:
        _remove_bad_elements(c)


def _remove_event_attrs(root):
    for e in root.iter():
        bad = [a for a in e.attrib
               if (a.split('}')[-1] if '}' in a else a).lower().startswith('on')]
        for a in bad:
            del e.attrib[a]


def fallback_svg(prompt=''):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" '
        'viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="gray"/>'
        '</svg>'
    )


_ok, _reason = validate_svg(fallback_svg())
print(f'Fallback SVG valid: {_ok} ({_reason}), length: {len(fallback_svg())} chars')

Fallback SVG valid: True (ok), length: 196 chars


---
## 3. Data Pipeline

Load SVG datasets from HuggingFace, normalize field names, and apply quality filters.

In [5]:
DATASET_CATALOG = {
    'OmniSVG/MMSVG-Icon': {
        'split': 'train',
        'prompt_fields': ['description', 'keywords', 'detail', 'prompt', 'text'],
        'svg_fields': ['svg', 'picosvg', 'completion', 'target'],
    },
    'xingxm/SVGX-SFT-1M': {
        'split': 'train',
        'prompt_fields': ['prompt', 'instruction', 'input', 'query'],
        'svg_fields': ['completion', 'output', 'svg', 'response'],
    },
    'thesantatitan/deepseek-svg-dataset': {
        'split': 'train',
        'prompt_fields': ['prompt', 'instruction', 'input', 'text', 'query'],
        'svg_fields': ['completion', 'output', 'svg', 'response', 'answer', 'target'],
    },
}

# Start with 1 source for baseline stability.
# Switch to SVGX-SFT-1M or add OmniSVG later for more data.
ACTIVE_SOURCES = ['thesantatitan/deepseek-svg-dataset']


def _pick_first(example, keys):
    for k in keys:
        if k in example and example[k] is not None:
            v = str(example[k]).strip()
            if v:
                return v
    return ''


_PROMPT_PREFIX_RE = re.compile(
    r'^(?:Generate|Create|Write|Make|Draw|Produce)\s+'
    r'(?:(?:the\s+|an?\s+)?(?:SVG|svg)\s+(?:code|image|graphic)\s+)?'
    r'(?:for\s+)?(?:an?\s+image\s+(?:that|which)\s+looks?\s+like[:\s]*)?',
    re.IGNORECASE,
)
_PROMPT_SUFFIX_RE = re.compile(
    r"[\.\s]*(?:Don'?t|do\s+not)\s+use\s+markdown.*$",
    re.IGNORECASE,
)


def _clean_prompt(prompt):
    """Strip instruction wrappers, keeping only the visual description."""
    prompt = _PROMPT_PREFIX_RE.sub('', prompt)
    prompt = _PROMPT_SUFFIX_RE.sub('', prompt)
    return prompt.strip()


_SVG_FIND = re.compile(r'<svg[^>]*>.*?</svg>', re.IGNORECASE | re.DOTALL)


def _extract_svg(text):
    """Extract the first <svg>...</svg> block from arbitrary text."""
    if not text:
        return ''
    m = _SVG_FIND.search(text)
    return m.group(0) if m else ''


def _normalize(example, prompt_fields, svg_fields):
    prompt = _pick_first(example, prompt_fields)
    prompt = _clean_prompt(prompt)
    svg_raw = _pick_first(example, svg_fields)
    svg = _extract_svg(svg_raw)

    if not svg:
        for val in example.values():
            if isinstance(val, str):
                svg = _extract_svg(val)
                if svg:
                    break

    if not svg:
        return {'prompt': '', 'svg': ''}

    # Normalize SVG to 256×256, strip bad elements, compact formatting
    svg = sanitize_svg(svg)

    if not prompt:
        for val in example.values():
            if isinstance(val, str):
                v = val.strip()
                if v and '<svg' not in v.lower() and len(v) < 500:
                    prompt = _clean_prompt(v)
                    break

    return {'prompt': prompt, 'svg': svg}


def _quality_filter(example):
    """Keep only samples with valid, reasonably-sized SVGs."""
    svg = example.get('svg', '')
    prompt = example.get('prompt', '')
    if not svg or not prompt:
        return False
    if len(svg) > CONFIG['max_svg_train_chars']:
        return False
    ok, _ = validate_svg(svg)
    return ok


def load_source(dataset_id, cfg, max_samples):
    print(f'Loading {dataset_id} ...')
    ds = load_dataset(dataset_id, split=cfg['split'])
    print(f'  Columns: {ds.column_names}')
    print(f'  Raw rows: {len(ds)}')
    if len(ds) > 0:
        sample = {k: (str(v)[:80] + '...') if isinstance(v, str) and len(str(v)) > 80 else v
                  for k, v in ds[0].items()}
        print(f'  Sample row: {sample}')
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: _normalize(ex, cfg['prompt_fields'], cfg['svg_fields']),
        remove_columns=ds.column_names,
        desc=f'normalizing {dataset_id}',
    )
    before = len(ds)
    ds = ds.filter(_quality_filter, desc=f'filtering {dataset_id}')
    print(f'  {dataset_id}: {len(ds)} usable rows (dropped {before - len(ds)})')
    return ds

In [6]:
# ── Load competition train.csv as primary data source ──
comp_df = pd.read_csv(CONFIG['train_data_path'])
print(f'Competition train.csv: {len(comp_df)} rows, columns={list(comp_df.columns)}')

comp_ds = Dataset.from_pandas(comp_df[['prompt', 'svg']])


def _normalize_comp(example):
    prompt = str(example.get('prompt', '')).strip()
    prompt = _clean_prompt(prompt)
    svg = sanitize_svg(str(example.get('svg', '')))
    return {'prompt': prompt, 'svg': svg}


comp_ds = comp_ds.map(_normalize_comp, remove_columns=comp_ds.column_names,
                      desc='normalizing train.csv')
before = len(comp_ds)
comp_ds = comp_ds.filter(_quality_filter, desc='filtering train.csv')
print(f'  Usable: {len(comp_ds)} rows (dropped {before - len(comp_ds)})')

max_samples = CONFIG['max_train_samples_per_source']
if max_samples and len(comp_ds) > max_samples:
    comp_ds = comp_ds.shuffle(seed=SEED).select(range(max_samples))
    print(f'  Subsampled to {len(comp_ds)} rows')

train_raw = comp_ds.shuffle(seed=SEED)
splits = train_raw.train_test_split(test_size=CONFIG['eval_size'], seed=SEED)
train_ds = splits['train']
eval_ds = splits['test']

print(f'\nTrain: {len(train_ds)} | Eval: {len(eval_ds)}')
if len(train_ds) > 0:
    print(f'Sample prompt: {train_ds[0]["prompt"][:120]}')
    print(f'Sample SVG (first 200): {train_ds[0]["svg"][:200]}')
else:
    raise RuntimeError(
        'Training set is empty after filtering. '
        'Check max_svg_train_chars / validate_svg settings.'
    )

Loading thesantatitan/deepseek-svg-dataset ...


README.md:   0%|          | 0.00/522 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.7M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/632k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

  Columns: ['prompt', 'completion', 'messages']
  Raw rows: 4750
  Sample row: {'prompt': 'Generate svg code for an image that looks like: a black background with a white ...', 'completion': '\n<think>\n\nAlright, so the user is asking for an SVG code that creates an image w...', 'messages': [{'content': 'Respond in the following format:\n<think>\n...\n</think>\n...\n<generated_svg>\n...\n</generated_svg>\n', 'role': 'system'}, {'content': "Generate svg code for an image that looks like: a black background with a white and red flower. Don't use markdown just give svg code\n", 'role': 'user'}, {'content': '\n<think>\n\nAlright, so the user is asking for an SVG code that creates an image with a black background and a white and red flower. They specified not to use markdown and just provide the SVG code. \n\nFirst, I need to structure the SVG properly. The basic structure includes the <svg> tag with width and height attributes. I\'ll set it to 400x400 for a square canvas.\n\nNext, the bac

normalizing thesantatitan/deepseek-svg-dataset:   0%|          | 0/4750 [00:00<?, ? examples/s]

filtering thesantatitan/deepseek-svg-dataset:   0%|          | 0/4750 [00:00<?, ? examples/s]

  thesantatitan/deepseek-svg-dataset: 3095 usable rows (dropped 1655)
Train: 3033 | Eval: 62
Sample prompt: Generate svg code for an image that looks like: a black background with a white and red flower. Don't use markdown just 
Sample SVG length: 791 chars


In [7]:
SYSTEM_PROMPT = (
    'You generate compact, valid SVG code from text descriptions. '
    'Output only a single <svg> element with xmlns="http://www.w3.org/2000/svg", '
    'width="256", height="256", viewBox="0 0 256 256". '
    'No explanation, no markdown — only SVG code.'
)


def format_sft(example):
    text = (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{example["prompt"]}<|im_end|>\n'
        '<|im_start|>assistant\n'
        f'{example["svg"]}<|im_end|>'
    )
    return {'text': text}


train_text = train_ds.map(format_sft, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft, remove_columns=eval_ds.column_names)

print('Formatted sample (first 500 chars):')
print(train_text[0]['text'][:500])

Map:   0%|          | 0/3033 [00:00<?, ? examples/s]

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

Formatted sample (first 500 chars):
<|im_start|>system
You generate compact, valid SVG code from text descriptions. Output only a single <svg> element with xmlns="http://www.w3.org/2000/svg", width="256", height="256", viewBox="0 0 256 256". No explanation, no markdown — only SVG code.<|im_end|>
<|im_start|>user
Generate svg code for an image that looks like: a black background with a white and red flower. Don't use markdown just give svg code<|im_end|>
<|im_start|>assistant
<svg width="200" height="200" viewBox="0 0 200 200">
  <


---
## 4. Model Setup — Qwen 2B + QLoRA (Unsloth)

In [8]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    target_modules=CONFIG['target_modules'],
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.11 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


---
## 5. Training

In [9]:
from transformers import TrainingArguments, Trainer as _HFTrainer
from trl import SFTTrainer

# ============ Fix: Unsloth / transformers >=4.46 compatibility ============
# transformers passes num_items_in_batch as int; Unsloth calls .mean() on it.
# Strategy A — class-level patch on Trainer.training_step (survives compiled cache)
_orig_trainer_step = _HFTrainer.training_step

def _safe_training_step(self, model, inputs, num_items_in_batch=None):
    if isinstance(num_items_in_batch, (int, float)):
        num_items_in_batch = torch.tensor(float(num_items_in_batch))
    return _orig_trainer_step(self, model, inputs, num_items_in_batch)

_HFTrainer.training_step = _safe_training_step
print("[patch] Trainer.training_step wrapped for int→tensor fix")
# ==========================================================================

training_args = TrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps',
    eval_steps=CONFIG['eval_steps'],
    save_steps=CONFIG['save_steps'],
    save_total_limit=2,
    report_to='none',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    seed=SEED,
)

_resp_tpl = '<|im_start|>assistant\n'
print(f'[info] response_template = {_resp_tpl!r}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field='text',
    max_seq_length=CONFIG['max_seq_length'],
    packing=False,
    args=training_args,
    response_template=_resp_tpl,
)

# Strategy B — disable num_items_in_batch from being passed at all
trainer.model_accepts_loss_kwargs = False
print("[patch] model_accepts_loss_kwargs = False  →  num_items_in_batch disabled")

train_result = trainer.train()
print(train_result)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[patch] Trainer.training_step wrapped for int→tensor fix


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/3033 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/62 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


[patch] model_accepts_loss_kwargs = False  →  num_items_in_batch disabled


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,033 | Num Epochs = 1 | Total steps = 48
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 32
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 32 x 1) = 64
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss,Validation Loss


TrainOutput(global_step=48, training_loss=0.012591260640571514, metrics={'train_runtime': 1643.0089, 'train_samples_per_second': 1.846, 'train_steps_per_second': 0.029, 'total_flos': 3.284721768357888e+16, 'train_loss': 0.012591260640571514, 'epoch': 1.0})


In [10]:
os.makedirs(CONFIG['output_dir'], exist_ok=True)
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f'Adapter + tokenizer saved to {CONFIG["output_dir"]}')

Adapter + tokenizer saved to /kaggle/working/qwen25_3b_svg_lora


In [11]:
import gc

# Free training artifacts — optimizer states, gradients, cached activations
del trainer, training_args
gc.collect()
torch.cuda.empty_cache()

allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
print(f"GPU memory after cleanup: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

GPU memory after cleanup: 2.58 GB allocated, 2.92 GB reserved


---
## Phase B: Inference & Submission

> Everything below this line can run **offline** on Kaggle.
> For the actual Kaggle code submission, create a separate notebook that loads the pre-trained adapter from a Kaggle dataset and only executes the cells below.

In [12]:
import gc, importlib
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Unsloth globally patches Qwen2* classes at import time (Cell 12).
# Reload the module to restore original, unpatched forward methods.
import transformers.models.qwen2.modeling_qwen2 as _qwen2
importlib.reload(_qwen2)
print('[fix] Reloaded transformers.models.qwen2.modeling_qwen2 (Unsloth patches removed)')

del model
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Use the clean (reloaded) Qwen2 class — NOT AutoModelForCausalLM,
# which may still reference the old patched class via its internal mapping.
base_model = _qwen2.Qwen2ForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model = PeftModel.from_pretrained(base_model, CONFIG['output_dir'])
tokenizer = AutoTokenizer.from_pretrained(CONFIG['output_dir'])
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()

alloc = torch.cuda.memory_allocated() / 1e9
print(f"Model reloaded (clean HF + LoRA). GPU mem: {alloc:.2f} GB")

SVG_EXTRACT_RE = re.compile(r'<svg.*?</svg>', flags=re.IGNORECASE | re.DOTALL)


def extract_svg(text):
    """Extract the first <svg>...</svg> block from model output."""
    m = SVG_EXTRACT_RE.search(text)
    return m.group(0).strip() if m else ''


def generate_svg(prompt):
    """Full pipeline: generate -> extract -> sanitize -> validate -> fallback.
    Returns (svg_string, status) where status tracks which layer failed."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    chat = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(chat, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=CONFIG['max_new_tokens'],
            do_sample=True,
            temperature=CONFIG['temperature'],
            top_p=CONFIG['top_p'],
            repetition_penalty=CONFIG['repetition_penalty'],
        )
    gen_ids = out[0][inputs['input_ids'].shape[-1]:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)
    print(f'  [debug] generated {len(gen_ids)} tokens, decoded len={len(decoded)}')
    print(f'  [debug] first 300 chars: {decoded[:300]}')
    raw = extract_svg(decoded)
    if not raw:
        return fallback_svg(prompt), 'extract_fail'
    cleaned = sanitize_svg(raw)
    if not cleaned:
        return fallback_svg(prompt), 'sanitize_fail'
    ok, reason = validate_svg(cleaned)
    if not ok:
        return fallback_svg(prompt), f'validate_fail:{reason}'
    return cleaned, 'ok'


def generate_svg_batch(prompts):
    """Batch inference — process multiple prompts in one forward pass."""
    bs = CONFIG.get('inference_batch_size', 4)
    all_results = []

    for i in range(0, len(prompts), bs):
        batch = prompts[i:i + bs]
        chats = []
        for p in batch:
            msgs = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': p},
            ]
            chats.append(tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True,
            ))

        inputs = tokenizer(
            chats, return_tensors='pt', padding=True,
            truncation=True, max_length=CONFIG['max_seq_length'],
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=CONFIG['max_new_tokens'],
                do_sample=True,
                temperature=CONFIG['temperature'],
                top_p=CONFIG['top_p'],
                repetition_penalty=CONFIG['repetition_penalty'],
            )

        input_len = inputs['input_ids'].shape[-1]
        for j, prompt in enumerate(batch):
            gen_ids = out[j][input_len:]
            decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)

            raw = extract_svg(decoded)
            if not raw:
                all_results.append((fallback_svg(prompt), 'extract_fail'))
                continue
            cleaned = sanitize_svg(raw)
            if not cleaned:
                all_results.append((fallback_svg(prompt), 'sanitize_fail'))
                continue
            ok, reason = validate_svg(cleaned)
            if not ok:
                all_results.append((fallback_svg(prompt), f'validate_fail:{reason}'))
                continue
            all_results.append((cleaned, 'ok'))

    return all_results

[fix] Reloaded transformers.models.qwen2.modeling_qwen2 (Unsloth patches removed)


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

Model reloaded (clean HF + LoRA). GPU mem: 3.01 GB


In [13]:
test_prompts = [
    'a simple red circle on white background',
    'a green tree with brown trunk',
    'a blue five-pointed star icon',
]

for p in test_prompts:
    svg, status = generate_svg(p)
    print(f'[{status}] len={len(svg):>5}  prompt="{p}"')
    print(f'  {svg[:150]}...')
    print()

Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[sanitize_fail] len=  196  prompt="a simple red circle on white background"
  <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circl...



Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[extract_fail] len=  196  prompt="a green tree with brown trunk"
  <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circl...

[sanitize_fail] len=  196  prompt="a blue five-pointed star icon"
  <svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circl...



---
## 7. Generate Submission

In [ ]:
test_df = pd.read_csv(CONFIG['test_prompts_path'])
print(f'Test prompts: {len(test_df)}')

all_prompts = test_df['prompt'].tolist()
all_ids = test_df['id'].tolist()
BS = CONFIG.get('inference_batch_size', 4)

rows = []
stats = Counter()
t0 = time.time()

for i in range(0, len(all_prompts), BS):
    batch_prompts = all_prompts[i:i + BS]
    batch_ids = all_ids[i:i + BS]
    batch_results = generate_svg_batch(batch_prompts)

    for pid, (svg, status) in zip(batch_ids, batch_results):
        stats[f'status:{status}'] += 1
        if status != 'ok':
            stats['fallback'] += 1
        else:
            stats['valid'] += 1
        stats['total_len'] += len(svg)
        rows.append({'id': pid, 'svg': svg})

    done = min(i + BS, len(all_prompts))
    if done % 50 < BS or done >= len(all_prompts):
        elapsed = time.time() - t0
        rate = done / max(elapsed, 1)
        eta = (len(all_prompts) - done) / max(rate, 0.01)
        print(f'  [{done}/{len(all_prompts)}] {elapsed:.0f}s '
              f'({rate:.2f} it/s, ETA {eta/60:.0f}min)  '
              f'valid={stats["valid"]}  fallback={stats["fallback"]}')

sub_df = pd.DataFrame(rows)
sub_df.to_csv(CONFIG['submission_path'], index=False)

elapsed_min = (time.time() - t0) / 60
n = len(sub_df)
print(f'\nSubmission saved: {CONFIG["submission_path"]}')
print(f'Total rows:    {n}')
print(f'Valid:         {stats["valid"]} ({100*stats["valid"]/max(n,1):.1f}%)')
print(f'Fallback:      {stats["fallback"]} ({100*stats["fallback"]/max(n,1):.1f}%)')
print(f'Avg SVG len:   {stats["total_len"]/max(n,1):.0f} chars')
print(f'Runtime:       {elapsed_min:.1f} min')

print(f'\n--- Failure distribution ---')
for k, v in sorted(stats.items()):
    if k.startswith('status:'):
        print(f'  {k}: {v}')

sub_df.head()

Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test prompts: 1000


Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

---
## Summary & Notes

**What this baseline does:**

- Loads SVG training data with quality filtering (valid XML, allowed tags, ≤4 000 chars)
- Fine-tunes Qwen 2B with QLoRA (`r=16`, 4-bit base, `packing=False` for clean sample boundaries)
- Generates SVGs with **conservative** decoding (`temperature=0.3`, `top_p=0.8`) to maximize validity
- Full post-processing chain: extract → sanitize → validate → fallback
- Tracks **per-layer failure reasons** (`extract_fail`, `sanitize_fail`, `validate_fail:<reason>`) for debugging

**Two-phase design:**

- **Phase A (Training):** Cells 1-15. Requires internet to load data & model. Saves adapter to `output_dir`.
- **Phase B (Inference):** Cells 16-20. Can run offline. For Kaggle Code Submission, create a separate notebook that only contains Phase B + loads the adapter from a Kaggle dataset.

**Key parameters to tune next:**

| Parameter | Current | Try |
|---|---|---|
| Data sources | 1 | 2-3 (add OmniSVG, SVGX subsets) |
| `max_train_samples` | 12 000 | 30 000-50 000 |
| `max_svg_train_chars` | 4 000 | 6 000-8 000 (with higher `max_seq_length`) |
| `lora_r` | 16 | 32, 64 |
| `lora_alpha` | 16 | 32 |
| `num_train_epochs` | 1 | 2-3 |
| `packing` | False | True (after confirming stable SVG output) |
| `temperature` | 0.3 | 0.5-0.7 (after valid rate ≥ 95%) |
| `max_new_tokens` | 768 | 1 024-1 536 |

**Checklist before submission:**

- [ ] Fixed random seeds everywhere
- [ ] `submission.csv` has columns `id`, `svg`
- [ ] All SVGs pass validity gate (fallback for failures)
- [ ] Failure distribution reviewed (no single failure mode dominates)
- [ ] Phase B notebook runs end-to-end on Kaggle without internet
- [ ] Dependency versions recorded (`requirements.txt`)
- [ ] GitHub repo + model weights link ready